In [7]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    balanced_accuracy_score,
    confusion_matrix
)

In [3]:
print(mlflow.__version__)

3.17.2


In [8]:
df = pd.read_excel("../data/cleaned_data.xlsx")

print("Shape:", df.shape)
print(df.columns.tolist())

Shape: (500, 17)
['Property_ID', 'Area_sqft', 'Bedrooms', 'Bathrooms', 'Floors', 'Location_Score', 'School_Score', 'Current_Price', 'Price_Year_1', 'Price_Year_2', 'Price_Year_3', 'Price_Year_4', 'Price_After_5_Years', 'Price_Category', 'Price_per_sqft', 'Area_per_Bedroom', 'Property_age']


In [6]:
df.shape

(500, 17)

In [9]:
df["Investment_Return"] = (
    (df["Price_After_5_Years"] - df["Current_Price"])
    / df["Current_Price"]
)

df["Good_Investment"] = (
    df["Investment_Return"] >= 0.30
).astype(int)

print(df["Good_Investment"].value_counts())

Good_Investment
1    326
0    174
Name: count, dtype: int64


In [10]:
regression_features = [
    "Area_sqft",
    "Bedrooms",
    "Bathrooms",
    "Floors",
    "Property_age",
    "Location_Score",
    "School_Score",
    "Price_per_sqft",
    "Area_per_Bedroom",
    "Current_Price"
]

X_reg = df[regression_features]
y_reg = df["Price_After_5_Years"]

print("Regression features:")
print(X_reg.columns.tolist())

Regression features:
['Area_sqft', 'Bedrooms', 'Bathrooms', 'Floors', 'Property_age', 'Location_Score', 'School_Score', 'Price_per_sqft', 'Area_per_Bedroom', 'Current_Price']


In [11]:
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

print("Training rows:", X_train_reg.shape[0])
print("Testing rows:", X_test_reg.shape[0])

Training rows: 400
Testing rows: 100


In [12]:
ridge_model = joblib.load("../models/ridge_model.pkl")
ridge_scaler = joblib.load("../models/scaler.pkl")

print("Ridge model loaded")
print("Regression scaler loaded")

Ridge model loaded
Regression scaler loaded


In [15]:
X_test_reg = X_test_reg[ridge_scaler.feature_names_in_]

X_test_reg_scaled = ridge_scaler.transform(X_test_reg)

In [17]:
y_pred_reg = ridge_model.predict(X_test_reg_scaled)

In [18]:
reg_mae = mean_absolute_error(y_test_reg, y_pred_reg)
reg_rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
reg_r2 = r2_score(y_test_reg, y_pred_reg)

print(f"MAE: {reg_mae:.2f}")
print(f"RMSE: {reg_rmse:.2f}")
print(f"R2 Score: {reg_r2:.2f}")

MAE: 72137.42
RMSE: 86560.49
R2 Score: 0.86


In [19]:
classification_features = [
    "Area_sqft",
    "Bedrooms",
    "Bathrooms",
    "Floors",
    "Location_Score",
    "School_Score",
    "Current_Price",
    "Price_per_sqft",
    "Area_per_Bedroom",
    "Property_age"
]

X_cls = df[classification_features]
y_cls = df["Good_Investment"]

print("Classification features:")
print(X_cls.columns.tolist())

Classification features:
['Area_sqft', 'Bedrooms', 'Bathrooms', 'Floors', 'Location_Score', 'School_Score', 'Current_Price', 'Price_per_sqft', 'Area_per_Bedroom', 'Property_age']


In [20]:
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

print("Training rows:", X_train_cls.shape[0])
print("Testing rows:", X_test_cls.shape[0])

Training rows: 400
Testing rows: 100


In [21]:
xgb_model = joblib.load("../models/xgb_classifier.pkl")
cls_scaler = joblib.load("../models/scaler_cls.pkl")

print("XGBoost model loaded")
print("Classification scaler loaded")

XGBoost model loaded
Classification scaler loaded


In [22]:
y_pred_cls = xgb_model.predict(X_test_cls)

In [23]:
cls_accuracy = accuracy_score(y_test_cls, y_pred_cls)
cls_precision = precision_score(y_test_cls, y_pred_cls)
cls_recall = recall_score(y_test_cls, y_pred_cls)
cls_f1 = f1_score(y_test_cls, y_pred_cls)
cls_f1_macro = f1_score(
    y_test_cls,
    y_pred_cls,
    average="macro"
)
cls_balanced_accuracy = balanced_accuracy_score(
    y_test_cls,
    y_pred_cls
)

print(f"Accuracy: {cls_accuracy:.4f}")
print(f"Precision: {cls_precision:.4f}")
print(f"Recall: {cls_recall:.4f}")
print(f"F1 Score: {cls_f1:.4f}")
print(f"F1 Macro: {cls_f1_macro:.4f}")
print(f"Balanced Accuracy: {cls_balanced_accuracy:.4f}")

Accuracy: 0.6300
Precision: 0.6795
Recall: 0.8154
F1 Score: 0.7413
F1 Macro: 0.5461
Balanced Accuracy: 0.5505


In [24]:
cm = confusion_matrix(y_test_cls, y_pred_cls)

print(cm)

[[10 25]
 [12 53]]


In [26]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")

mlflow.set_experiment("Real Estate Investment Advisor")

print("MLflow experiment configured.")

2026/11/03 00:16:10 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/03 00:16:10 INFO mlflow.store.db.utils: Updating database tables
2026/11/03 00:16:12 INFO mlflow.tracking.fluent: Experiment with name 'Real Estate Investment Advisor' does not exist. Creating a new experiment.


MLflow experiment configured.


In [27]:
with mlflow.start_run(run_name="Final_Ridge_Regression"):

    mlflow.log_param("model", "Ridge Regression")
    mlflow.log_param("alpha", 1)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)

    mlflow.log_metric("MAE", reg_mae)
    mlflow.log_metric("RMSE", reg_rmse)
    mlflow.log_metric("R2", reg_r2)

    mlflow.log_artifact("../models/ridge_model.pkl")
    mlflow.log_artifact("../models/scaler.pkl")

    print("Regression run logged successfully.")

Regression run logged successfully.


In [28]:
with mlflow.start_run(run_name="Final_XGBoost_Classification"):

    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_estimators", xgb_model.get_params()["n_estimators"])
    mlflow.log_param("max_depth", xgb_model.get_params()["max_depth"])
    mlflow.log_param("learning_rate", xgb_model.get_params()["learning_rate"])
    mlflow.log_param("test_size", 0.2)
    mlflow.log_param("random_state", 42)
    mlflow.log_param("investment_threshold", 0.30)

    mlflow.log_metric("Accuracy", cls_accuracy)
    mlflow.log_metric("Precision", cls_precision)
    mlflow.log_metric("Recall", cls_recall)
    mlflow.log_metric("F1", cls_f1)
    mlflow.log_metric("F1_Macro", cls_f1_macro)
    mlflow.log_metric("Balanced_Accuracy", cls_balanced_accuracy)

    mlflow.log_artifact("../models/xgb_classifier.pkl")
    mlflow.log_artifact("../models/scaler_cls.pkl")

    print("Classification run logged successfully.")

Classification run logged successfully.


In [29]:
experiment = mlflow.get_experiment_by_name(
    "Real Estate Investment Advisor"
)

print("Experiment name:", experiment.name)
print("Experiment ID:", experiment.experiment_id)
print("Tracking URI:", mlflow.get_tracking_uri())

Experiment name: Real Estate Investment Advisor
Experiment ID: 1
Tracking URI: sqlite:///../mlflow.db


In [31]:
print(runs.columns.tolist())

['run_id', 'experiment_id', 'status', 'artifact_uri', 'start_time', 'end_time', 'metrics.F1_Macro', 'metrics.Precision', 'metrics.Accuracy', 'metrics.F1', 'metrics.Balanced_Accuracy', 'metrics.Recall', 'metrics.RMSE', 'metrics.R2', 'metrics.MAE', 'params.max_depth', 'params.investment_threshold', 'params.learning_rate', 'params.test_size', 'params.random_state', 'params.model', 'params.n_estimators', 'params.alpha', 'tags.mlflow.source.type', 'tags.mlflow.runName', 'tags.mlflow.source.name', 'tags.mlflow.user']


In [32]:
runs = mlflow.search_runs(
    experiment_names=["Real Estate Investment Advisor"]
)

runs[[
    "run_id",
    "tags.mlflow.runName",
    "status",
    "start_time"
]]

,run_id,tags.mlflow.runName,status,start_time
0,4441e2be642d4557a1560ae0203bb63a,Final_XGBoost_Classification,FINISHED,2026-11-02 18:46:41.763000+00:00
1,ef395bf72763456bb7a8001b4f01efa2,Final_Ridge_Regression,FINISHED,2026-11-02 18:46:35.384000+00:00
